# CDX index -> single-name constituents (Bloomberg BDS test)

Goal: deduce which single names are in each CDX index via `xbbg` BDS, then map to `market_cds_ticker` in `teams.structured_credit_abhutra.cds_transform_interpolated`.

Requires a running Bloomberg Terminal session (bbcomm) on this machine.

Note: if printing fails with a cp1252 UnicodeEncodeError, launch Jupyter with `PYTHONIOENCODING=utf-8`.

In [ ]:
import sys
import pandas as pd
import xbbg

# own utils: not pip-installed, so add the repo root to sys.path
BBG_UTILS_PATH = 'S:/Structured Credit/Matt Stuff/claude_repos/bbg_utils'
if BBG_UTILS_PATH not in sys.path:
    sys.path.insert(0, BBG_UTILS_PATH)
from bbg_utils import bds_mm, bdp_mm, bdh_mm

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
print('xbbg', xbbg.__version__)

## 1. Connectivity check

In [ ]:
bdp_mm('SPX Index', 'PX_LAST')

## 2. Pull constituents with BDS
`INDX_MEMBERS` works on the generic CDX tickers (verified: IG returns 125 rows incl. extra/defaulted lines, HY 100). Swap in a specific series (e.g. `CDX IG CDSI S45 5Y Corp`) for a historical roster.

In [ ]:
INDEX_TICKERS = [
    'CDX IG CDSI GEN 5Y Corp',
    'CDX HY CDSI GEN 5Y Corp',
]
FIELD = 'INDX_MEMBERS'

# BDS columns come back generic; names below are inferred from the data, confirm them.
COLMAP = {
    'Member Ticker and Exchange Code': 'member_name',
    'column   1': 'coupon_or_weight',
    'column   2': 'currency',
    'column   3': 'seniority',
    'column   4': 'bbg_cds_ticker',
    'column   5': 'sp_id',
}

members = (bds_mm(INDEX_TICKERS, FIELD)
           .rename(columns=COLMAP)
           .rename(columns={'security': 'index_ticker'}))
print(members.groupby('index_ticker').size())
members.head(10)

## 3. Sanity checks
IG should be ~125 names and HY ~100. Check for duplicates and odd rows before mapping.

In [ ]:
members.groupby('index_ticker').agg(n=('member_name','size'), n_unique=('member_name','nunique'))

In [ ]:
# Map BBG constituents -> Market_CDS_Ticker via the mappings file.
# cds_mappings_final.csv: BBG_Short_Ticker == members.bbg_cds_ticker, CDS_Ticker == market_cds_ticker in Databricks.
MAPPINGS_PATH = 'S:/Structured Credit/Matt Stuff/cds_mappings/cds_mappings_final.csv'

mappings = (pd.read_csv(MAPPINGS_PATH, usecols=['CDS_Ticker', 'BBG_Short_Ticker'],
                        dtype=str, encoding='cp1252')   # file isn't utf-8
            .dropna(subset=['BBG_Short_Ticker'])
            .drop_duplicates('BBG_Short_Ticker')        # keys are unique today; guards a many-to-one blowup
            .rename(columns={'CDS_Ticker': 'Market_CDS_Ticker'}))

members = (members.drop(columns=['Market_CDS_Ticker'], errors='ignore')
           .merge(mappings, how='left', left_on='bbg_cds_ticker', right_on='BBG_Short_Ticker')
           .drop(columns='BBG_Short_Ticker'))

print(members.groupby('index_ticker')['Market_CDS_Ticker'].agg(n='size', mapped='count'))
print('Unmapped:')
display(members.loc[members['Market_CDS_Ticker'].isna(), ['index_ticker', 'member_name', 'bbg_cds_ticker']])
members.head(10)

In [ ]:
from cdx_members import get_index_members

ig = get_index_members('CDX IG CDSI GEN 5Y Corp')
print(len(ig), 'members;', ig['Market_CDS_Ticker'].isna().sum(), 'unmapped')
ig.head(10)

## 4. Next steps
- Identify which column maps to RED code / Markit ticker (`bbg_cds_ticker` vs `sp_id`).
- Join to `cds_transform_interpolated` on `red_code` or `markit_ticker`.
- Build the index -> names mapping table.